# Sarvam-30B → GGUF Conversion Attempt

**Goal:** Convert `sarvamai/sarvam-30b` to GGUF format for local inference via Ollama/llama.cpp/LM Studio.

**Context:** Sarvam uses sigmoid routing (not softmax) in its MoE architecture. This is non-standard and may not be supported by llama.cpp's conversion script yet.

**Architecture:**
- 30B total params, 2.4B active
- 19 layers (1 dense + 18 MoE)
- 128 routed experts + 1 shared, top-6 routing
- Sigmoid routing (the blocker)

Read more: [mtrajan.substack.com/p/sarvam-open-is-not-sovereign](https://mtrajan.substack.com/p/sarvam-open-is-not-sovereign)

## Step 1: Setup

In [ ]:
!pip install -q huggingface_hub hf_transfer sentencepiece protobuf transformers torch numpy gguf

# Check what we're working with
import shutil
total, used, free = shutil.disk_usage("/content")
free_gb = free / (1024**3)
print(f"\nDisk: {free_gb:.1f} GB free")
print(f"Need: ~65 GB for model + ~65 GB for GGUF conversion")

if free_gb < 100:
    print(f"\n⚠️  Only {free_gb:.1f} GB free. Will use Google Drive as overflow.")
    USE_DRIVE = True
else:
    print(f"\n✅ {free_gb:.1f} GB free. Enough for local storage.")
    USE_DRIVE = False

!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader 2>/dev/null || echo 'No GPU (CPU-only mode)'

In [ ]:
import os, subprocess

LLAMA_CPP_DIR = "/content/llama.cpp"

# Clone llama.cpp
if not os.path.exists(LLAMA_CPP_DIR):
    !git clone --depth 1 https://github.com/ggml-org/llama.cpp {LLAMA_CPP_DIR}
else:
    !cd {LLAMA_CPP_DIR} && git pull

# Build conversion tools (CPU is fine — GPU not needed for conversion)
!cd {LLAMA_CPP_DIR} && \
    cmake -B build -DBUILD_SHARED_LIBS=OFF -DGGML_CUDA=OFF -DCMAKE_BUILD_TYPE=Release && \
    cmake --build build --config Release -j$(nproc) --target llama-quantize llama-cli

# Verify build
for tool in ["llama-quantize", "llama-cli"]:
    path = f"{LLAMA_CPP_DIR}/build/bin/{tool}"
    if os.path.exists(path):
        print(f"✅ {tool} built")
    else:
        print(f"❌ {tool} NOT found — build may have failed")

import os, shutil

# --- Disk strategy ---
total, used, free = shutil.disk_usage("/content")
free_gb = free / (1024**3)

if free_gb < 100:
    print(f"⚠️  {free_gb:.1f} GB free — mounting Google Drive for storage")
    from google.colab import drive
    drive.mount('/content/drive')
    MODEL_DIR = "/content/drive/MyDrive/sarvam-gguf/sarvam-30b"
    OUTPUT_DIR = "/content/drive/MyDrive/sarvam-gguf/output"
    CACHE_DIR = "/content/hf_cache"  # cache stays on fast local disk
else:
    print(f"✅ {free_gb:.1f} GB free — using local storage")
    MODEL_DIR = "/content/sarvam-30b"
    OUTPUT_DIR = "/content/output"
    CACHE_DIR = "/content/hf_cache"

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
os.environ["HF_HOME"] = CACHE_DIR

print(f"\nModel dir: {MODEL_DIR}")
print(f"Output dir: {OUTPUT_DIR}")

# --- Download ---
from huggingface_hub import snapshot_download

if os.path.exists(f"{MODEL_DIR}/config.json"):
    print("\n✅ Model already downloaded, skipping.")
else:
    print("\n⬇️  Downloading sarvamai/sarvam-30b (~60GB)...")
    try:
        snapshot_download(
            repo_id="sarvamai/sarvam-30b",
            local_dir=MODEL_DIR,
            ignore_patterns=["*.bin", "*.h5", "*.msgpack", "original/**"],
            resume_download=True,  # resume if interrupted
        )
        print("✅ Download complete!")
    except Exception as e:
        print(f"❌ Download failed: {e}")
        print("\nTroubleshooting:")
        print("  1. Check disk: !df -h /content")
        print("  2. Try mounting Drive and re-run this cell")
        print("  3. Try: Runtime → Manage sessions → terminate, then reconnect")
        raise

# Clean up HF cache to free disk for conversion
!rm -rf {CACHE_DIR}/hub 2>/dev/null

print(f"\nModel size: ", end="")
!du -sh {MODEL_DIR}
print("Remaining disk: ", end="")
!df -h /content | awk 'NR==2{print $4, "free"}'

In [ ]:
import os
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

from huggingface_hub import snapshot_download

snapshot_download(
    repo_id="sarvamai/sarvam-30b",
    local_dir="/content/sarvam-30b",
    ignore_patterns=["*.bin"],  # prefer safetensors
)

import json

with open(f"{MODEL_DIR}/config.json") as f:
    config = json.load(f)

print(json.dumps(config, indent=2))
print(f"\n{'='*50}")
print(f"Model type:     {config.get('model_type', 'unknown')}")
print(f"Architectures:  {config.get('architectures', [])}")

# Find MoE-specific keys
moe_keys = [k for k in config.keys() if any(x in k.lower() for x in ['expert', 'moe', 'router', 'routing'])]
print(f"\nMoE config keys:")
for k in moe_keys:
    print(f"  {k}: {config[k]}")
print(f"{'='*50}")

In [ ]:
import json

with open("/content/sarvam-30b/config.json") as f:
    config = json.load(f)

print(json.dumps(config, indent=2))
print(f"\nModel type: {config.get('model_type', 'unknown')}")
print(f"Architectures: {config.get('architectures', [])}")

# Find MoE-specific keys
moe_keys = [k for k in config.keys() if any(x in k.lower() for x in ['expert', 'moe', 'router', 'routing'])]
print(f"MoE config keys: {moe_keys}")
for k in moe_keys:
    print(f"  {k}: {config[k]}")

LLAMA_CPP_DIR = "/content/llama.cpp"

# Does convert_hf_to_gguf.py know about Sarvam?
print("=== Sarvam/sigmoid references in conversion script ===")
!grep -n -i 'sarvam\|sigmoid\|SarvamMoe' {LLAMA_CPP_DIR}/convert_hf_to_gguf.py || echo '❌ No Sarvam support in conversion script'

print("\n=== Sarvam references in gguf-py ===")
!grep -rn -i 'sarvam\|sigmoid' {LLAMA_CPP_DIR}/gguf-py/ 2>/dev/null | head -10 || echo '❌ No Sarvam support in gguf-py'

print("\n=== Sarvam references in llama.cpp C++ source ===")
!grep -rn -i 'sarvam\|sigmoid.*rout' {LLAMA_CPP_DIR}/src/ {LLAMA_CPP_DIR}/include/ 2>/dev/null | head -10 || echo '❌ No Sarvam support in C++ source'

# List all supported model architectures for reference
print("\n=== Supported architectures in convert_hf_to_gguf.py ===")
!grep -oP 'class \w+Model' {LLAMA_CPP_DIR}/convert_hf_to_gguf.py 2>/dev/null | sort || \
 grep 'class.*Model' {LLAMA_CPP_DIR}/convert_hf_to_gguf.py | head -40

In [ ]:
# Does convert_hf_to_gguf.py know about Sarvam?
!grep -n 'sarvam\|sigmoid\|SarvamMoe' /content/llama.cpp/convert_hf_to_gguf.py || echo '\n>>> No Sarvam support found in conversion script'
print("---")
!grep -rn 'sarvam\|sigmoid' /content/llama.cpp/gguf-py/ 2>/dev/null | head -10 || echo '>>> No Sarvam support in gguf-py'
print("---")
!grep -rn 'sarvam\|sigmoid' /content/llama.cpp/src/ /content/llama.cpp/include/ 2>/dev/null | head -10 || echo '>>> No Sarvam support in llama.cpp source'

import os, time

LLAMA_CPP_DIR = "/content/llama.cpp"
GGUF_F16 = f"{OUTPUT_DIR}/sarvam-30b-f16.gguf"

print("🔄 Attempting GGUF conversion...")
print(f"   Input:  {MODEL_DIR}")
print(f"   Output: {GGUF_F16}")
print()

start = time.time()
result = !python3 {LLAMA_CPP_DIR}/convert_hf_to_gguf.py \
    {MODEL_DIR} \
    --outfile {GGUF_F16} \
    --outtype f16 \
    2>&1

# Print output
for line in result:
    print(line)

elapsed = time.time() - start
print(f"\n⏱️  Took {elapsed/60:.1f} minutes")

if os.path.exists(GGUF_F16):
    size_gb = os.path.getsize(GGUF_F16) / (1024**3)
    print(f"\n{'='*50}")
    print(f"✅ SUCCESS! GGUF created: {size_gb:.2f} GB")
    print(f"{'='*50}")
    CONVERSION_OK = True
else:
    print(f"\n{'='*50}")
    print(f"❌ CONVERSION FAILED")
    print(f"{'='*50}")
    print("\nThis likely means llama.cpp doesn't support Sarvam's architecture yet.")
    print("See Step 8 for diagnosis.")
    CONVERSION_OK = False

# Save log either way
with open(f"{OUTPUT_DIR}/conversion_log.txt", "w") as f:
    f.write("\n".join(result))

In [ ]:
# Check if it worked
if CONVERSION_OK:
    print(f"✅ GGUF exists at: {GGUF_F16}")
    print(f"   Size: {os.path.getsize(GGUF_F16)/(1024**3):.2f} GB")
    print("\nProceeding to quantization (Step 6)...")
else:
    print("❌ No GGUF produced. Scroll up for errors, or skip to Step 8 for diagnosis.")

In [ ]:
# Check if it worked
import os
gguf_path = "/content/sarvam-30b-f16.gguf"
if os.path.exists(gguf_path):
    size_gb = os.path.getsize(gguf_path) / (1024**3)
    print(f"SUCCESS! GGUF created: {size_gb:.2f} GB")
    print("Proceeding to quantization...")
else:
    print("CONVERSION FAILED")
    print("\nFull log:")
    !cat /content/conversion_log.txt

import os

LLAMA_CPP_DIR = "/content/llama.cpp"
QUANTIZE = f"{LLAMA_CPP_DIR}/build/bin/llama-quantize"

if not CONVERSION_OK:
    print("⏭️  Skipping — no GGUF to quantize.")
elif not os.path.exists(QUANTIZE):
    print("❌ llama-quantize not found. Rebuild llama.cpp.")
else:
    # Check disk before quantizing
    import shutil
    _, _, free = shutil.disk_usage(OUTPUT_DIR)
    free_gb = free / (1024**3)
    print(f"Disk free: {free_gb:.1f} GB\n")

    methods = ["q8_0", "q4_k_m"]
    if free_gb < 40:
        print(f"⚠️  Low disk — only doing Q4_K_M")
        methods = ["q4_k_m"]

    for method in methods:
        out = f"{OUTPUT_DIR}/sarvam-30b-{method}.gguf"
        print(f"🔄 Quantizing to {method}...")
        !{QUANTIZE} {GGUF_F16} {out} {method}
        if os.path.exists(out):
            size_gb = os.path.getsize(out) / (1024**3)
            print(f"✅ {method}: {size_gb:.2f} GB\n")
        else:
            print(f"❌ {method} failed\n")

    # Delete F16 to save disk if quants succeeded
    if os.path.exists(f"{OUTPUT_DIR}/sarvam-30b-q4_k_m.gguf"):
        print(f"🗑️  Removing F16 GGUF to save disk...")
        os.remove(GGUF_F16)

    print("\n📁 Output files:")
    for f in sorted(os.listdir(OUTPUT_DIR)):
        path = os.path.join(OUTPUT_DIR, f)
        if os.path.isfile(path):
            size = os.path.getsize(path)
            if size > 1024*1024:
                print(f"  {f}: {size/(1024**3):.2f} GB")
            else:
                print(f"  {f}: {size} bytes")

In [ ]:
import os
gguf_path = "/content/sarvam-30b-f16.gguf"

if os.path.exists(gguf_path):
    for method in ["q8_0", "q4_k_m"]:
        out = f"/content/sarvam-30b-{method}.gguf"
        print(f"\nQuantizing to {method}...")
        !cd /content/llama.cpp && ./build/bin/llama-quantize {gguf_path} {out} {method}
        if os.path.exists(out):
            print(f"  → {os.path.getsize(out)/(1024**3):.2f} GB")
else:
    print("No GGUF to quantize. See Step 5 for errors.")

# === Upload to HuggingFace ===
# Uncomment below, set your HF token and repo name, then run.

# from huggingface_hub import HfApi, login
# login()  # will prompt for token
# api = HfApi()
#
# # Create the repo
# api.create_repo("YOUR_USERNAME/sarvam-30b-GGUF", repo_type="model", exist_ok=True)
#
# # Upload all GGUFs
# import glob
# for gguf in glob.glob(f"{OUTPUT_DIR}/*.gguf"):
#     name = os.path.basename(gguf)
#     print(f"Uploading {name}...")
#     api.upload_file(
#         path_or_fileobj=gguf,
#         path_in_repo=name,
#         repo_id="YOUR_USERNAME/sarvam-30b-GGUF",
#         repo_type="model",
#     )
#     print(f"  ✅ Done")

print("ℹ️  Uncomment this cell and set YOUR_USERNAME to upload.")

In [ ]:
# Uncomment and fill in to upload
# from huggingface_hub import HfApi
# api = HfApi()
# api.upload_file(
#     path_or_fileobj="/content/sarvam-30b-q4_k_m.gguf",
#     path_in_repo="sarvam-30b-q4_k_m.gguf",
#     repo_id="YOUR_USERNAME/sarvam-30b-GGUF",
#     repo_type="model",
# )

import json

with open(f"{MODEL_DIR}/config.json") as f:
    config = json.load(f)

model_type = config.get("model_type", "unknown")
archs = config.get("architectures", [])

print(f"{'='*60}")
print(f"DIAGNOSIS")
print(f"{'='*60}")
print(f"\nSarvam model_type:    '{model_type}'")
print(f"Sarvam architectures: {archs}")

# What does llama.cpp support?
LLAMA_CPP_DIR = "/content/llama.cpp"
print(f"\nSupported model classes in convert_hf_to_gguf.py:")
!grep -oP 'class (\w+Model)' {LLAMA_CPP_DIR}/convert_hf_to_gguf.py 2>/dev/null | sort -u | head -50

# Check if model_type is in the supported list
result = !grep -c -i '{model_type}' {LLAMA_CPP_DIR}/convert_hf_to_gguf.py 2>/dev/null
count = int(result[0]) if result else 0

print(f"\n{'='*60}")
if count > 0:
    print(f"✅ '{model_type}' found {count} times in conversion script")
    print("   Conversion should work (or there's a different issue)")
else:
    print(f"❌ '{model_type}' NOT found in conversion script")
    print(f"   This is the blocker. llama.cpp doesn't know this architecture.")
    print(f"\n   The fix:")
    print(f"   1. Wait for llama.cpp PR #20275 to merge")
    print(f"   2. Or patch convert_hf_to_gguf.py using vLLM PR #33942 as reference")
    print(f"   3. Key challenge: sigmoid routing in MoE layer")
print(f"{'='*60}")

print(f"\nReferences:")
print(f"  vLLM PR #33942 (merged):     github.com/vllm-project/vllm/pull/33942")
print(f"  llama.cpp PR #20275 (open):  github.com/ggml-org/llama.cpp/pull/20275")
print(f"  Analysis: mtrajan.substack.com/p/sarvam-open-is-not-sovereign")

In [ ]:
# What model_type does Sarvam report?
import json
with open("/content/sarvam-30b/config.json") as f:
    config = json.load(f)

model_type = config.get("model_type", "unknown")
print(f"Sarvam model_type: '{model_type}'")

# What model_types does convert_hf_to_gguf.py support?
print("\nSupported model types in convert_hf_to_gguf.py:")
!grep -oP '"model_type".*?"\K[^"]+' /content/llama.cpp/convert_hf_to_gguf.py 2>/dev/null || \
 grep 'model_type' /content/llama.cpp/convert_hf_to_gguf.py | head -30

print(f"\n{'='*60}")
print("DIAGNOSIS:")
print(f"Sarvam uses model_type='{model_type}'")
print("If this type is not in the list above, that's the blocker.")
print(f"{'='*60}")
print("\nReferences:")
print("- vLLM PR #33942 (merged): Sarvam MoE support")
print("- llama.cpp PR #20275: sarvam_moe architecture (pending)")
print("- Blog: mtrajan.substack.com/p/sarvam-open-is-not-sovereign")